In [ ]:
%pip install wquantiles

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
from scipy.stats import trim_mean
from statsmodels import robust
import wquantiles
import seaborn as sns
import matplotlib.pylab as plt

# 1. Structured and Rectangular Data

Dibuku membedakan structured data menjadi beberapa bentuk. Untuk banyak pekerjaan data science, bentuk yang paling umum adalah **rectangular data**.

Dalam rectangular data:
- setiap **row** merupakan record/observation,
- setiap **column** merupakan feature/variable.



In [ ]:
# Table 1-2
state = pd.read_csv('state.csv')
print(state.head(8))

# 2. Estimates of Location

**Location** digunakan untuk menggambarkan posisi pusat data.

## Mean

Untuk data \(x_1, x_2, ..., x_n\):

$$
\bar{x} = \frac{1}{n}\sum_{i=1}^{n}x_i
$$

Mean mudah dihitung tetapi sensitif terhadap nilai ekstrem.

## Trimmed Mean

Trimmed mean membuang sejumlah nilai terkecil dan terbesar sebelum menghitung mean. Tujuannya adalah mengurangi pengaruh extreme values.

## Median

Median adalah nilai tengah setelah data diurutkan. Median lebih robust terhadap outlier dibandingkan mean.

## Weighted Mean

Jika setiap observasi mempunyai bobot \(w_i\):

$$
\bar{x}_w =
\frac{\sum_{i=1}^{n} w_i x_i}
{\sum_{i=1}^{n} w_i}
$$

Weighted mean berguna ketika observasi mempunyai tingkat kepentingan atau ukuran yang berbeda.

In [ ]:
state['Population'].mean()

In [ ]:
trim_mean(state['Population'], 0.1)

In [ ]:
state['Population'].median()

In [ ]:
np.average(state['Murder.Rate'], weights=state['Population'])

In [ ]:
wquantiles.median(state['Murder.Rate'], weights=state['Population'])

### Interpretation

Perbedaan antara ordinary mean dan weighted mean menunjukkan bahwa cara kita memberikan bobot pada observasi dapat mengubah hasil.

Untuk murder rate, population dapat digunakan sebagai weight karena negara bagian dengan populasi besar mewakili jumlah penduduk yang jauh lebih besar daripada negara bagian dengan populasi kecil.

# 3. Estimates of Variability

**Variability** atau **dispersion** menunjukkan seberapa menyebar nilai data.

Beberapa ukuran yang dibahas:

### Deviation

Deviation suatu observasi terhadap mean:

$$
x_i - \bar{x}
$$

### Variance

Untuk sample:

$$
s^2 =
\frac{1}{n-1}
\sum_{i=1}^{n}(x_i-\bar{x})^2
$$

### Standard Deviation

$$
s = \sqrt{s^2}
$$

Standard deviation menggunakan satuan yang sama dengan data asli.

### IQR

Interquartile range:

$$
IQR = Q_3-Q_1
$$

### Median Absolute Deviation from the Median

MAD menggunakan jarak absolut setiap nilai terhadap median. Ukuran ini lebih robust terhadap outlier dibanding standard deviation.

In [ ]:
print(state.head(8))

In [ ]:
state['Population'].std()

In [ ]:
state['Population'].quantile(0.75) - state['Population'].quantile(0.25)

In [ ]:
robust.scale.mad(state['Population'])

# 4. Percentiles and Boxplots

**Percentile** adalah nilai yang memiliki persentase tertentu dari data berada di bawahnya.

Contoh:
- 25th percentile = Q1,
- 50th percentile = median,
- 75th percentile = Q3.

Percentile berguna untuk melihat keseluruhan distribusi, termasuk bagian ekor/tails.

## Boxplot

Boxplot memberikan ringkasan visual distribusi:
- bagian bawah box = 25th percentile,
- garis di dalam box = median,
- bagian atas box = 75th percentile,
- whiskers menunjukkan rentang bulk data,
- titik di luar whiskers dapat muncul sebagai potential outliers.


In [ ]:
state['Murder.Rate'].quantile([0.05, 0.25, 0.5, 0.75, 0.95])

In [ ]:
ax = (state['Population']/1_000_000).plot.box()
ax.set_ylabel('Population (millions)')

### Interpretasi

Pada data state, median population berada sekitar beberapa juta jiwa dan terdapat beberapa negara bagian dengan populasi jauh lebih tinggi. Boxplot membuat ketidaksimetrian dan nilai ekstrem tersebut mudah terlihat.

# 5. Frequency Tables and Histograms

**Frequency table** membagi rentang nilai menjadi interval atau **bins**, kemudian menghitung berapa observasi yang berada di setiap interval.

Histogram merupakan visualisasi frequency table dengan:
- nilai/bin pada sumbu x,
- count atau proportion pada sumbu y.

In [ ]:
binnedPopulation = pd.cut(state['Population'], 10)
binnedPopulation.value_counts()

In [ ]:
binnedPopulation.name = 'binnedPopulation'
df = pd.concat([state, binnedPopulation], axis=1)
df = df.sort_values(by='Population')

groups = []
for group, subset in df.groupby(by='binnedPopulation', observed=False):
    groups.append({
        'BinRange': group,
        'Count': len(subset),
        'States': ','.join(subset.Abbreviation)
    })
print(pd.DataFrame(groups))

In [ ]:
ax = (state['Population'] / 1_000_000).plot.hist()
ax.set_xlabel('Population (millions)')

### Interpretasi

Frequency table memberikan versi numerik dari histogram. Dengan melihat jumlah observasi di setiap bin, kita dapat memahami apakah data terkonsentrasi pada rentang tertentu atau memiliki ekor panjang.

# 6. Density Plots and KDE

Density plot merupakan representasi distribusi yang lebih halus dibandingkan histogram.

Kernel Density Estimate (KDE) dapat ditulis:

$$
\hat{f}(x) =
\frac{1}{nh}
\sum_{i=1}^{n}
K\left(\frac{x-x_i}{h}\right)
$$

di mana \(h\) merupakan bandwidth yang mengontrol tingkat smoothing.

Density plot berguna untuk melihat bentuk distribusi tanpa bergantung langsung pada batas-batas bin histogram.

In [ ]:
ax = state['Murder.Rate'].plot.hist(density=True, xlim=[0,12], bins=range(1,12))
state['Murder.Rate'].plot.density(ax=ax)
ax.set_xlabel('Murder Rate (per 100,000)')

### Interpretasi

Histogram menunjukkan frequency, sedangkan density plot menunjukkan bentuk distribusi sebagai kurva halus. Total area di bawah density curve bernilai 1.

# 7. Binary and Categorical Data

Untuk categorical data, jumlah mentah sering lebih mudah dipahami jika dikonversi menjadi **proportion** atau **percentage**.

Untuk binary variable:

$$
\text{Proportion} =
\frac{\text{jumlah kejadian}}
{\text{jumlah observasi}}
$$

Hasilnya dapat dikalikan 100 untuk memperoleh percentage.

In [ ]:
dfw = pd.read_csv('dfw_airline.csv')
dfw

In [ ]:
ax = dfw.transpose().plot.bar(figsize=(4, 4), legend=False)
ax.set_xlabel('Cause of delay')
ax.set_ylabel('Count')

# 8. Exploring Two or More Variables — Correlation

**Correlation coefficient** mengukur tingkat asosiasi antara dua paired variables.

Pearson correlation berada pada rentang:

$$
-1 \le r \le 1
$$

- nilai positif → kedua variabel cenderung bergerak searah,
- nilai negatif → cenderung bergerak berlawanan,
- mendekati 0 → asosiasi linear lemah.

Correlation bukan ukuran yang tepat untuk semua bentuk hubungan, terutama hubungan non-linear.

In [ ]:
sp500_sym = pd.read_csv('sp500_sectors.csv')
sp500_px = pd.read_csv('sp500_data.csv.gz', index_col=0)

In [ ]:
# Table 1-7
telecomSymbols = sp500_sym[sp500_sym['sector'] == 'telecommunications_services']['symbol']
telecom = sp500_px.loc[sp500_px.index >= '2012-07-01', telecomSymbols]
telecom.corr()

## Correlation Matrix dan Heatmap

Digunakan ETF untuk menunjukkan correlation matrix dalam bentuk visual. Heatmap membantu ketika jumlah variabel cukup banyak sehingga tabel correlation sulit dibaca.

In [ ]:
etfs = sp500_px.loc[sp500_px.index > '2012-07-01',
                    sp500_sym[sp500_sym['sector'] == 'etf']['symbol']]
sns.heatmap(etfs.corr(), vmin=-1, vmax=1,
            cmap=sns.diverging_palette(20, 220, as_cmap=True))

### Interpretasi

Diagonal correlation matrix selalu bernilai 1 karena setiap variabel berkorelasi sempurna dengan dirinya sendiri. Pasangan dengan correlation positif tinggi cenderung bergerak searah.

# 9. Scatterplots

Scatterplot digunakan untuk melihat hubungan antara dua variabel numerik.

Setiap titik merepresentasikan satu observasi.

Scatterplot penting karena correlation coefficient hanya merangkum hubungan secara numerik, sedangkan scatterplot memungkinkan kita melihat pola, outlier, dan bentuk hubungan secara visual.

In [1]:
ax = telecom.plot.scatter(x='T', y='VZ', figsize=(4, 4), marker='$\u25EF$')
ax.set_xlabel('ATT (T)')
ax.set_ylabel('Verizon (VZ)')
ax.axhline(0, color='grey', lw=1)
ax.axvline(0, color='grey', lw=1)

NameError: name 'telecom' is not defined

### Interpretasi

Scatterplot memperlihatkan kecenderungan hubungan positif antara return ATT dan Verizon. Titik banyak terkumpul di sekitar nol, dengan kecenderungan kedua saham bergerak ke arah yang sama pada banyak hari.

# 10. Hexagonal Binning and Contours

Jika jumlah data sangat besar, scatterplot dapat menjadi sulit dibaca karena terlalu banyak titik yang bertumpuk.

**Hexagonal binning** mengelompokkan titik ke dalam sel berbentuk hexagon dan menampilkan kepadatan data.

**Contour plot / density plot 2D** juga dapat digunakan untuk melihat kepadatan dua variabel numerik.

In [ ]:
kc_tax = pd.read_csv('kc_tax.csv.gz')
kc_tax0 = kc_tax.loc[(kc_tax.TaxAssessedValue < 750000) &
                     (kc_tax.SqFtTotLiving > 100) &
                     (kc_tax.SqFtTotLiving < 3500), :]

In [ ]:
ax = kc_tax0.plot.hexbin(x='SqFtTotLiving', y='TaxAssessedValue',
                         gridsize=30, sharex=False)
ax.set_xlabel('Finished Square Feet')
ax.set_ylabel('Tax Assessed Value')

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
sns.kdeplot(data=kc_tax0.sample(10000), x='SqFtTotLiving',
            y='TaxAssessedValue', ax=ax)
ax.set_xlabel('Finished Square Feet')
ax.set_ylabel('Tax Assessed Value')

### Interpretasi

Hexbin memperlihatkan area yang memiliki konsentrasi observasi tinggi tanpa menampilkan seluruh titik satu per satu. KDE dua dimensi memberikan representasi kepadatan yang lebih halus.

# 11. Two Categorical Variables — Contingency Tables

**Contingency table** merangkum jumlah observasi berdasarkan dua categorical variables.

- `grade` = grade personal loan,
- `status` = status loan.

Pivot table dapat digunakan untuk memperoleh count dan persentase.

In [ ]:
lc_loans = pd.read_csv('lc_loans.csv')
lc_loans

In [ ]:
crosstab = lc_loans.pivot_table(index='grade', columns='status',
                                aggfunc=lambda x: len(x), margins=True)
crosstab

In [ ]:
df = crosstab.loc['A':'G',:].copy()
df.loc[:,'Charged Off':'Late'] = df.loc[:,'Charged Off':'Late'].div(df['All'],
                                                                     axis=0)
df['All'] = df['All'] / sum(df['All'])
perc_crosstab = df
perc_crosstab

### Interpretasi

Contingency table memungkinkan kita melihat pola status loan pada setiap grade. Persentase baris membantu membandingkan komposisi status antar-grade, bukan hanya jumlah absolut.

# 12. Categorical and Numeric Data

Ketika satu variabel bersifat categorical dan variabel lainnya numeric, boxplot dapat digunakan untuk membandingkan distribusi numeric variable antar kategori.

Violin plot menambahkan informasi mengenai bentuk density distribusi.

In [ ]:
airline_stats = pd.read_csv('airline_stats.csv')
airline_stats.head()

In [ ]:
ax = airline_stats.boxplot(by='airline', column='pct_carrier_delay')
ax.set_xlabel('')
ax.set_ylabel('Daily % of Delayed Flights')
plt.suptitle('')

In [ ]:
ax = sns.violinplot(airline_stats.airline, airline_stats.pct_carrier_delay,
                    inner='quartile', color='white')
ax.set_xlabel('')
ax.set_ylabel('Daily % of Delayed Flights')

### Interpretasi

Boxplot memperlihatkan median, quartile, dan outlier. Violin plot menambahkan informasi tentang bentuk density distribution sehingga beberapa pola distribusi yang tidak jelas pada boxplot dapat terlihat.

# 13. Multiple Variables

EDA tidak hanya dilakukan pada satu variabel.

Secara umum:

- **Univariate** → satu variabel.
- **Bivariate** → dua variabel.
- **Multivariate** → lebih dari dua variabel.

Pemilihan visualisasi bergantung pada tipe variabel:

| Kombinasi | Contoh |
|---|---|
| Numeric × Numeric | Scatterplot, hexbin |
| Categorical × Categorical | Contingency table |
| Categorical × Numeric | Boxplot, violin plot |

Tujuan akhirnya adalah memperoleh pemahaman mengenai struktur, distribusi, outlier, dan hubungan dalam dataset sebelum melakukan modeling.

In [ ]:
zip_codes = [98188, 98105, 98108, 98126]
kc_tax_zip = kc_tax0.loc[kc_tax0.ZipCode.isin(zip_codes),:]
kc_tax_zip

def hexbin(x, y, color, **kwargs):
    cmap = sns.light_palette(color, as_cmap=True)
    plt.hexbin(x, y, gridsize=25, cmap=cmap, **kwargs)

g = sns.FacetGrid(kc_tax_zip, col='ZipCode', col_wrap=2)
g.map(hexbin, 'SqFtTotLiving', 'TaxAssessedValue',
      extent=[0, 3500, 0, 700000])
g.set_axis_labels('Finished Square Feet', 'Tax-Assessed Value')
g.set_titles('Zip code {col_name:.0f}')

### Interpretasi

Facet memungkinkan hubungan dua variabel numerik dilihat secara terpisah berdasarkan kategori tambahan, yaitu zip code. Ini membantu mengungkap perbedaan distribusi yang dapat tersembunyi ketika seluruh data digabungkan.